# Lesson 1: Python Fundamentals

**Week 3 · Data Engineering Course**

---

This lesson is not a syntax tour. It is an explanation of how Python actually works — the object model, memory behaviour, the iterator protocol, and the performance characteristics of common data structures. Understanding these internals is what separates someone who copies Python from someone who writes it confidently.

Every concept in this lesson has a direct consequence for data engineering work. Those consequences are called out explicitly.

**Sections:**
1. Python's Object Model
2. Mutability and Its Traps
3. The Iterator Protocol and Generators
4. Strings at the Byte Level
5. Functions as First-Class Objects
6. Data Structures and Their Costs

In [ ]:
import sys
import tracemalloc
import timeit
from copy import copy, deepcopy
print(f'Python {sys.version}')

---

## 1.1 Python's Object Model

In Python, **everything is an object**. That includes integers, strings, functions, classes, and modules. Every object has three things:

- An **identity** — its address in memory. `id(x)` returns it.
- A **type** — what kind of object it is. `type(x)` returns it.
- A **value** — its content.

Understanding identity is critical because Python never implicitly copies objects. When you write `a = b`, you are not making a copy of `b`. You are giving the same object a second name.

In [ ]:
# Every value has an identity, type, and value
x = 42
print(f'value: {x}')
print(f'type:  {type(x)}')
print(f'id:    {id(x)}')  # memory address of the integer object 42

# Assignment gives a second name to the same object
a = [1, 2, 3]
b = a              # b points to the SAME list object as a
print(f'\na is b: {a is b}')  # True — same object in memory
print(f'id(a):  {id(a)}')
print(f'id(b):  {id(b)}')   # identical

# Consequence: mutating b mutates a, because they are the same object
b.append(4)
print(f'\na after b.append(4): {a}')  # a is affected

### `is` vs `==`

`is` tests **identity** — whether two names refer to the same object in memory.
`==` tests **equality** — whether two objects have the same value.

They are completely different tests. For `None`, always use `is`. For values, use `==`.

Why? Because `==` can be overridden by any class. `None is None` is always unambiguously true. `value == None` might do something unexpected if the object defines `__eq__`.

In [ ]:
# is vs == — these are not the same question
a = [1, 2, 3]
b = [1, 2, 3]   # same VALUE, different OBJECT

print(f'a == b:  {a == b}')   # True  (same value)
print(f'a is b:  {a is b}')   # False (different objects in memory)
print(f'id(a):   {id(a)}')
print(f'id(b):   {id(b)}')    # different

# None is a singleton — there is only one None object in the entire Python process
x = None
y = None
print(f'\nx is y:  {x is y}')   # True — there is only one None
print(f'id(None): {id(None)}')
print(f'id(x):    {id(x)}')   # same address

# The right way to check for None in a pipeline
def process(value):
    if value is None:      # correct
        return 'missing'
    return str(value)

print(process(None))
print(process(0))   # 0 is falsy but is NOT None

### Small Integer Caching

CPython (the standard Python interpreter) caches integers in the range -5 to 256. Any time you reference one of these values, Python reuses the same object rather than creating a new one. Outside that range, each assignment creates a new object.

This is an implementation detail — your code should never rely on it. But it explains why `is` comparisons on integers sometimes surprise you.

In [ ]:
# Cached range: same object every time
a = 100
b = 100
print(f'100: a is b = {a is b}')   # True — cached

a = 257
b = 257
print(f'257: a is b = {a is b}')   # False — outside cache, new objects

# Practical consequence: NEVER use `is` to compare integers or strings
# Use == for values, is only for None, True, False
count = 256
if count is 256:    # this happens to work but is wrong practice
    print('cached match')
if count == 256:    # always correct
    print('value match')

# sys.getrefcount reveals how many names point to an object
# The count includes the temporary reference created by getrefcount itself
print(f'\nref count of 100: {sys.getrefcount(100)}')
# High count because 100 is cached and reused everywhere in the interpreter

---

## 1.2 Mutability and Its Traps

**Immutable** objects cannot be changed after creation: `int`, `float`, `str`, `tuple`, `bool`, `None`.
**Mutable** objects can be changed: `list`, `dict`, `set`.

When you write `s = s + ' suffix'` on a string, you are not modifying `s`. You are creating a **new** string object and rebinding the name `s` to it. The original string is unchanged (and eventually garbage collected).

For data engineering this matters in two places:
1. Strings are safe to pass around — nothing can mutate your data underneath you.
2. Lists and dicts passed to functions are NOT copies — mutations inside the function affect the caller.

In [ ]:
# String immutability: id changes on every reassignment
s = 'ELECTRONICS'
print(f'original id: {id(s)}')
s = s.strip().lower()
print(f'new id:      {id(s)}')  # different object — original string still exists in memory

# List mutability: id stays the same after mutation
lst = ['a', 'b', 'c']
print(f'\nlist id before: {id(lst)}')
lst.append('d')
print(f'list id after:  {id(lst)}')  # same object — modified in place

# Consequence for functions: lists passed in can be mutated
def append_total(rows, total):
    rows.append({'total': total})  # mutates the caller's list

my_rows = [{'product': 'Laptop'}]
append_total(my_rows, 1200)
print(f'\nmy_rows after call: {my_rows}')  # the caller sees the change

# To avoid this, pass a copy
def safe_append(rows, total):
    rows = list(rows)  # local copy
    rows.append({'total': total})
    return rows

my_rows = [{'product': 'Laptop'}]
result = safe_append(my_rows, 1200)
print(f'original unchanged: {my_rows}')

In [ ]:
# The Mutable Default Argument — classic Python trap
# Default argument values are evaluated ONCE when the function is defined,
# not each time the function is called. A mutable default is shared across all calls.

def add_row_BAD(row, accumulator=[]):
    accumulator.append(row)
    return accumulator

print(add_row_BAD({'id': 1}))  # [{'id': 1}]
print(add_row_BAD({'id': 2}))  # [{'id': 1}, {'id': 2}] — the list persisted!
print(add_row_BAD({'id': 3}))  # keeps growing — bug

# The fix: use None as the sentinel, create a new list inside
def add_row(row, accumulator=None):
    if accumulator is None:
        accumulator = []
    accumulator.append(row)
    return accumulator

print(add_row({'id': 1}))  # [{'id': 1}]
print(add_row({'id': 2}))  # [{'id': 2}] — fresh list each time

In [ ]:
# Shallow copy vs deep copy
# copy() copies the outer container but not nested objects
import copy

original = [{'name': 'Alice', 'scores': [90, 85]}, {'name': 'Bob', 'scores': [70, 80]}]
shallow = copy.copy(original)
deep = copy.deepcopy(original)

# Modify a nested list in the shallow copy
shallow[0]['scores'].append(95)

print(f'original[0][scores]: {original[0]["scores"]}')
# [90, 85, 95] — shallow copy shares the nested list object!
print(f'shallow[0][scores]:  {shallow[0]["scores"]}')
# [90, 85, 95] — same nested list

deep[0]['scores'].append(100)
print(f'\noriginal after deep change: {original[0]["scores"]}')
# [90, 85, 95] — deep copy is independent

---

## 1.3 The Iterator Protocol and Generators

Every `for` loop in Python calls two things:

1. `iter(obj)` — calls `obj.__iter__()`, returns an **iterator** object
2. Repeatedly calls `next(iterator)` — calls `iterator.__next__()`, returns the next value
3. When done, `__next__` raises `StopIteration` — the loop catches it and stops

This is the iterator protocol. Lists, dicts, files, CSV readers, database cursors, and generators all implement it. That is why you can `for row in csv_reader` and `for row in database_cursor` with identical syntax.

A **generator** is a function that implements the iterator protocol lazily — it produces one value at a time on demand, rather than building the entire sequence in memory first. For data engineering, this is critical: you cannot load a 10 GB CSV file into a list. You can process it with a generator.

In [ ]:
# Prove what a for loop does at the protocol level
numbers = [10, 20, 30]

# This is what `for n in numbers` actually does:
iterator = iter(numbers)           # calls numbers.__iter__()
print(type(iterator))              # <class 'list_iterator'>
print(next(iterator))              # 10 — calls iterator.__next__()
print(next(iterator))              # 20
print(next(iterator))              # 30
try:
    print(next(iterator))          # raises StopIteration
except StopIteration:
    print('StopIteration raised — loop would end here')

# Any class that implements __iter__ and __next__ works in a for loop
class CountUp:
    def __init__(self, limit):
        self.current = 0
        self.limit = limit

    def __iter__(self):
        return self   # this object is its own iterator

    def __next__(self):
        if self.current >= self.limit:
            raise StopIteration
        value = self.current
        self.current += 1
        return value

for n in CountUp(4):
    print(n, end=' ')
print()

In [ ]:
# Generator functions: yield instead of return
# Python turns any function with a yield statement into a generator
# The function body does NOT run until you call next()

def read_rows_lazy(data):
    '''Yield one row at a time. No list is ever built in memory.'''
    for row in data:
        yield row

# The generator object is created instantly, nothing runs yet
rows = ['row1', 'row2', 'row3']
gen = read_rows_lazy(rows)
print(type(gen))              # <class 'generator'>

# Values are produced on demand
print(next(gen))              # row1 — now the function body runs up to yield
print(next(gen))              # row2
print(next(gen))              # row3
# next(gen) would raise StopIteration

# A generator that simulates reading a large CSV
def read_csv_stream(lines):
    '''Strip and split each line. No list of all lines is kept.'''
    headers = None
    for line in lines:
        parts = line.strip().split(',')
        if headers is None:
            headers = parts
            continue
        yield dict(zip(headers, parts))

fake_csv = ['order_id,product,total', '1001,Laptop,1200', '1002,Mouse,35']
for row in read_csv_stream(fake_csv):
    print(row)

In [ ]:
# Memory comparison: list vs generator
# tracemalloc measures how much memory your code allocates

tracemalloc.start()

# Option A: build a list of 1 million integers in memory
list_version = [x * 2 for x in range(1_000_000)]
_, list_peak = tracemalloc.get_traced_memory()
tracemalloc.reset_peak()

# Option B: a generator expression — computes values lazily
gen_version = (x * 2 for x in range(1_000_000))
_, gen_peak = tracemalloc.get_traced_memory()

tracemalloc.stop()

print(f'List peak memory:      {list_peak / 1024 / 1024:.1f} MB')
print(f'Generator peak memory: {gen_peak / 1024:.1f} KB')
print()
print('The generator uses essentially no memory because it holds no values.')
print('It computes each value when next() is called, then discards it.')

In [ ]:
# Chaining generators: a streaming pipeline
# Each stage is a generator that pulls from the previous stage
# No intermediate lists are created

def source(rows):
    '''Stage 1: emit one row at a time.'''
    for row in rows:
        yield row

def strip_whitespace(rows):
    '''Stage 2: strip whitespace from string values.'''
    for row in rows:
        yield {k: v.strip() if isinstance(v, str) else v for k, v in row.items()}

def drop_invalid(rows):
    '''Stage 3: drop rows where quantity is not positive.'''
    for row in rows:
        try:
            qty = float(row.get('quantity', 0))
            if qty > 0:
                yield row
        except (ValueError, TypeError):
            pass  # silently drop unparseable rows

raw = [
    {'product': '  Laptop  ', 'quantity': '2'},
    {'product': 'Mouse', 'quantity': '-1'},  # invalid
    {'product': '  Keyboard  ', 'quantity': '1'},
]

# Connect the stages — nothing runs until we iterate
pipeline = drop_invalid(strip_whitespace(source(raw)))

for row in pipeline:
    print(row)
# Memory usage at any moment: just the current row in each stage

---

## 1.4 Strings at the Byte Level

Python strings are Unicode — each character is a code point, not a byte. `len('café')` is 4, not 5, even though `'café'` encoded as UTF-8 is 5 bytes. The distinction between text and bytes becomes critical when reading files: you must know (or detect) the encoding, or you get garbled data or crashes.

For string cleaning at scale, `str.translate()` is significantly faster than chaining multiple `.replace()` calls because it does one pass through the string. For pattern-based extraction, the `re` module with **compiled patterns** avoids recompiling the regex on every call.

In [ ]:
# Unicode: code points, not bytes
s = 'caf\u00e9'   # \u00e9 is the code point for e with accent
print(f'string:   {s}')
print(f'len():    {len(s)}')         # 4 characters
print(f'as bytes: {s.encode("utf-8")}')  # b'caf\xc3\xa9' — 5 bytes (e with accent is 2 bytes in UTF-8)
print(f'as bytes: {len(s.encode("utf-8"))}')

# Encoding converts str -> bytes. Decoding converts bytes -> str.
bytes_data = b'caf\xc3\xa9'
back_to_str = bytes_data.decode('utf-8')
print(f'\ndecoded: {back_to_str}')  # café

# What happens with the wrong encoding
try:
    wrong = bytes_data.decode('ascii')
except UnicodeDecodeError as e:
    print(f'\nDecode error: {e}')
    print('This is what you see when a file was written with one encoding and read with another.')

In [ ]:
# str.translate() for bulk character removal — one pass, faster than chained .replace()
# Build a translation table: map characters to None (delete) or replacements

price_str = '$1,200.00'

# Naive approach: three separate passes through the string
price_naive = price_str.replace('$', '').replace(',', '').strip()

# translate() approach: one pass
table = str.maketrans({'$': None, ',': None})
price_fast = price_str.translate(table).strip()

print(f'naive:     {price_naive}')
print(f'translate: {price_fast}')
print(f'equal: {price_naive == price_fast}')

# Timing comparison on 100,000 calls
import timeit
stmt_naive = "'$1,200.00'.replace('$', '').replace(',', '')"
stmt_table = """
table = str.maketrans({'$': None, ',': None})
'$1,200.00'.translate(table)
"""

time_naive = timeit.timeit(stmt_naive, number=100_000)
time_table = timeit.timeit(stmt_table, number=100_000)
print(f'\n100k calls — naive: {time_naive:.3f}s  translate: {time_table:.3f}s')

In [ ]:
import re

# Compiled regex: compile once, reuse many times
# re.compile() parses the pattern into a state machine
# Without compile(), Python re-parses the pattern on every call

# Pattern to extract parts of a date like 2023-07-14
DATE_PATTERN = re.compile(r'(?P<year>\d{4})-(?P<month>\d{2})-(?P<day>\d{2})')

dates = ['2023-07-14', '2024-01-01', 'not-a-date', '2023-12-31']
for s in dates:
    m = DATE_PATTERN.match(s)
    if m:
        print(f'{s}  ->  year={m.group("year")} month={m.group("month")} day={m.group("day")}')
    else:
        print(f'{s}  ->  no match')

# re.sub() with a function — apply transformation to each match
MIXED_CASE = re.compile(r'[a-z]+')

def to_upper(match):
    return match.group(0).upper()

result = MIXED_CASE.sub(to_upper, 'home & kitchen')
print(f'\nreplaced: {result}')  # HOME & KITCHEN

---

## 1.5 Functions as First-Class Objects

In Python, functions are objects. You can:
- Store a function in a variable or list
- Pass a function as an argument to another function
- Return a function from a function
- Assign two names to the same function

This is the foundation for decorators, callbacks, and configuration-driven pipelines. In data engineering, it means you can build a list of cleaning functions and apply them in sequence, without hardcoding the sequence.

In [ ]:
# Functions are objects — store, pass, return
def strip_value(s):
    return str(s).strip()

def to_title(s):
    return str(s).title()

def remove_dollar(s):
    return str(s).replace('$', '')

# A pipeline of functions stored in a list
cleaning_steps = [strip_value, to_title]

value = '  electronics  '
for step in cleaning_steps:
    value = step(value)
print(value)  # Electronics

# Apply a list of functions to every value in a column
def apply_pipeline(values, steps):
    '''Apply a sequence of functions to each value in a list.'''
    result = []
    for v in values:
        for step in steps:
            v = step(v)
        result.append(v)
    return result

categories = ['  ELECTRONICS  ', 'clothing', '  HOME & KITCHEN  ']
steps = [strip_value, to_title]
print(apply_pipeline(categories, steps))

In [ ]:
# Closures: a function that remembers variables from the scope where it was created

def make_typo_fixer(typo_map):
    '''Returns a function that applies typo_map corrections.'''
    def fix(value):
        return typo_map.get(str(value).strip().title(), str(value).strip().title())
    return fix
# fix() is a closure: it closes over typo_map from make_typo_fixer

typos = {
    'Electrnics': 'Electronics',
    'Home & Kithen': 'Home & Kitchen',
}
fix_category = make_typo_fixer(typos)

print(fix_category('Electrnics'))       # Electronics
print(fix_category('  electronics  '))  # Electronics (stripped and title-cased first)
print(fix_category('Clothing'))         # Clothing (no change)

# You can create different fixers for different typo maps
fix_region = make_typo_fixer({'Nrth': 'North', 'Sth': 'South'})
print(fix_region('Nrth'))              # North

In [ ]:
from functools import partial
from typing import Optional

# Type hints: annotations that document intent
# Python does NOT enforce type hints at runtime — they are documentation only
# But type checkers (mypy, pyright) and IDEs use them

def clean_numeric(
    value: str | float | int,
    remove_chars: str = '$,',
    default: Optional[float] = None
) -> Optional[float]:
    '''Convert a potentially dirty value to float, returning default if it fails.'''
    try:
        cleaned = str(value)
        for ch in remove_chars:
            cleaned = cleaned.replace(ch, '')
        return float(cleaned.strip())
    except (ValueError, TypeError):
        return default

print(clean_numeric('$1,200.00'))      # 1200.0
print(clean_numeric('N/A'))            # None
print(clean_numeric('N/A', default=0)) # 0

# functools.partial: pre-fill some arguments of a function
# Useful when you want a specialised version of a general function
clean_euro = partial(clean_numeric, remove_chars='\u20ac.')
# \u20ac is the euro sign €
print(clean_euro('\u20ac1.200'))       # 1200.0

---

## 1.6 Data Structures and Their Costs

Choosing the right data structure changes the performance of a pipeline by orders of magnitude. The key question is always: **what operations do you do most frequently?**

| Structure | Lookup | Insert (end) | Insert (front) | Membership test |
|-----------|--------|--------------|----------------|-----------------|
| `list` | O(n) | O(1) | O(n) | O(n) |
| `dict` | O(1) | O(1) | — | O(1) for keys |
| `set` | — | O(1) | — | O(1) |

O(1) means constant time regardless of size. O(n) means the operation slows down linearly as the container grows.

For deduplication in a pipeline: use a `set`, not a list. For lookup tables (category corrections, region mappings): use a `dict`, not a loop.

In [ ]:
# Prove the O(n) vs O(1) difference for membership testing
import timeit

size = 100_000
haystack_list = list(range(size))
haystack_set = set(range(size))
needle = size - 1   # worst case for a list: must scan all elements

time_list = timeit.timeit(lambda: needle in haystack_list, number=10_000)
time_set  = timeit.timeit(lambda: needle in haystack_set,  number=10_000)

print(f'List membership (10k searches): {time_list:.4f}s')
print(f'Set membership  (10k searches): {time_set:.4f}s')
print(f'Set is ~{time_list / time_set:.0f}x faster')
print()
print('Use a set any time you need to check membership or deduplicate.')
print('Use a dict any time you need to look up a value by key.')

In [ ]:
# Deduplication with a set vs a list
import csv
from pathlib import Path

data_path = Path('../data/raw/sales_q1.csv')

# Deduplicate rows by order_id
rows_list = []
with open(data_path, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    rows_list = list(reader)

# Naive: scan accumulator list on every row — O(n^2) total
def dedup_naive(rows, key):
    seen = []
    result = []
    for row in rows:
        if row[key] not in seen:    # O(n) scan each time
            seen.append(row[key])
            result.append(row)
    return result

# Efficient: set lookup — O(n) total
def dedup_fast(rows, key):
    seen = set()
    result = []
    for row in rows:
        if row[key] not in seen:    # O(1) lookup
            seen.add(row[key])
            result.append(row)
    return result

naive = dedup_naive(rows_list, 'order_id')
fast  = dedup_fast(rows_list, 'order_id')
print(f'Original: {len(rows_list)} rows')
print(f'Deduped (naive): {len(naive)} rows')
print(f'Deduped (fast):  {len(fast)} rows')
print(f'Results match: {naive == fast}')

In [ ]:
# Dict as a lookup table — O(1) correction of category typos at any scale

TYPO_MAP = {
    'electrnics': 'Electronics',
    'home & kithen': 'Home & Kitchen',
    'electronics': 'Electronics',
    'clothing': 'Clothing',
    'home & kitchen': 'Home & Kitchen',
}

def correct_category(value: str) -> str:
    normalised = str(value).strip().lower()
    return TYPO_MAP.get(normalised, str(value).strip().title())
# .get(key, default) returns the value if found, otherwise default
# This avoids a KeyError and handles unknown categories gracefully

test_values = ['ELECTRONICS', 'Electrnics', '  clothing  ', 'Home & Kithen', 'Unknown']
for v in test_values:
    print(f'{v!r:25} -> {correct_category(v)}')

In [ ]:
from collections import defaultdict, Counter

# Counter: count occurrences in one pass
with open('../data/raw/sales_q1.csv', newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    categories = [row['category'].strip() for row in reader]

raw_counts = Counter(categories)
print('Raw category distribution:')
for cat, count in raw_counts.most_common():
    print(f'  {cat!r}: {count}')

# defaultdict: avoids the if-key-not-in-dict boilerplate
# Groups rows by category without checking if the key exists
with open('../data/raw/sales_q1.csv', newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    by_category = defaultdict(list)
    for row in reader:
        cat = correct_category(row['category'])
        by_category[cat].append(row['order_id'])

print('\nOrder IDs by category (normalised):')
for cat, ids in sorted(by_category.items()):
    print(f'  {cat}: {len(ids)} orders')

---

## Summary

**What you now know — and why it matters:**

| Concept | Data engineering consequence |
|---------|-----------------------------|
| Everything is an object; assignment is binding | Passing a list to a function lets that function mutate your data. Always be explicit about whether you want a copy. |
| `is` vs `==` | Use `is None`, not `== None`. Use `==` for every value comparison. |
| Mutable default arguments | Never use `def f(rows=[])`. Use `None` and create the list inside. |
| Shallow vs deep copy | `copy.copy()` is not safe for nested structures. Know which one you need. |
| Iterator protocol | Files, CSV readers, database cursors are all iterators. You can chain them with generator functions without building intermediate lists. |
| Generators save memory | A 10 GB file processed with a generator chain uses kilobytes of RAM. The same file loaded as a list crashes the process. |
| `str.translate()` | Faster than chained `.replace()` for character removal — matters when cleaning millions of rows. |
| Compiled regex | `re.compile(pattern)` once; reuse. Never put `re.sub(pattern, ...)` inside a row-by-row loop without compiling first. |
| Closures | Use `make_X(config)` patterns to create pre-configured cleaning functions. More readable than global variables. |
| Set for membership, dict for lookup | These are O(1). List scan is O(n). The difference is invisible at 100 rows and catastrophic at 10 million. |